# Практикум 2. Чтение энергетических данных и первичный контроль качества

В этом практикуме рассматривается первая задача, без которой невозможен дальнейший анализ: нужно открыть тематические файлы, понять их структуру и убедиться, что данные действительно пригодны для интерпретации и моделирования.

## Почему этот этап принципиален

Даже хороший алгоритм не исправит ошибки в исходной таблице. Если пропустить проверку структуры, типов или дубликатов, последующие графики и метрики будут описывать не изучаемый процесс, а дефекты входных данных.

## Что будет сделано в практикуме

- будет открыт основной набор по устойчивости режима;
- будет составлен его краткий паспорт;
- будет выполнен первичный контроль качества;
- будут прочитаны тематические фрагменты в форматах `CSV`, `XLSX`, `TXT` и `JSON`;
- будет показано, как формат файла влияет на способ чтения, но не меняет логику анализа.

## Данные и инструменты

| Источник | Роль в практикуме |
| --- | --- |
| `electrical_grid_stability.csv` | основной набор для контроля качества и дальнейшей классификации |
| `grid_stability_fragment.csv` | пример локального тематического `CSV` |
| `combined_cycle_power_plant_fragment.xlsx` | пример чтения `XLSX` |
| `household_power_fragment.txt` | пример текстового файла с разделителем `;` |
| `dataset_catalog.json` | пример служебного файла с метаданными |
| `pandas` | чтение, проверка структуры и формирование сводных таблиц |

## Ход работы

### Шаг 1. Открыть основной набор данных
Сначала рассмотрим тематическую таблицу, на которой в следующих практикумах будет строиться классификационная постановка.

### Шаг 2. Оценить структуру и качество
На этом этапе нужно установить размерность, типы признаков, долю пропусков и наличие дубликатов.

### Шаг 3. Сопоставить несколько форматов файлов
После этого можно перейти к реальным локальным примерам `CSV`, `XLSX`, `TXT` и `JSON`, чтобы увидеть различия в чтении и представлении данных.

In [ ]:
import pandas as pd
from IPython.display import HTML, display

from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note
from src.project_paths import sample_data_dir

BASE_DIR = sample_data_dir()
DATA_PATH = BASE_DIR / "classification" / "electrical_grid_stability.csv"
FORMAT_DIR = BASE_DIR / "formats"

display_stage_note(
    "Шаг 1. Основной тематический набор",
    "Откроем локальную учебную таблицу и сразу посмотрим, как она выглядит в первых строках.",
)

df = pd.read_csv(DATA_PATH)
display_frame(df.head(6))

## Что видно после первого чтения

Таблица уже на первом экране показывает важную особенность: здесь присутствуют числовые признаки режима и целевое поле устойчивости. Значит, дальше необходимо не только оценить чистоту данных, но и не перепутать признаки с целевой меткой.

In [ ]:
passport = pd.DataFrame(
    [
        {
            "Источник": "Electrical Grid Stability Simulated Data",
            "Строк": len(df),
            "Столбцов": df.shape[1],
            "Числовых полей": df.select_dtypes(include="number").shape[1],
            "Категориальных полей": df.select_dtypes(exclude="number").shape[1],
        }
    ]
)
display_frame(passport, decimals=0)

## Паспорт набора данных

Такой краткий паспорт полезен не только в учебной работе. Он позволяет быстро восстановить контекст: откуда взяты данные, насколько они объемны и какие типы признаков в них преобладают.

In [ ]:
display_stage_note(
    "Шаг 2. Контроль качества",
    "Теперь проверим признаки на пропуски, типы и повторы строк, чтобы зафиксировать риски еще до моделирования.",
)

profile = pd.DataFrame(
    {
        "Признак": df.columns,
        "Тип": df.dtypes.astype(str).values,
        "Доля пропусков": df.isna().mean().round(4).values,
        "Число уникальных значений": df.nunique(dropna=False).values,
    }
)

display_frame(profile)

duplicate_count = int(df.duplicated().sum())
class_distribution = (
    df["stabf"].value_counts().rename_axis("Класс").reset_index(name="Число наблюдений")
)
display_frame(class_distribution, decimals=0)

display_metric_table(
    {
        "дублирующихся строк": float(duplicate_count),
        "максимальная доля пропусков": float(profile["Доля пропусков"].max()),
        "классов обнаружено": float(class_distribution.shape[0]),
    },
    decimals=0,
)

## Как интерпретировать профиль качества

Если доля пропусков мала или равна нулю, это еще не означает, что набор полностью готов к моделированию. Однако такая проверка быстро показывает, есть ли критические структурные проблемы. Распределение целевой метки дополнительно помогает понять, можно ли позже доверять простой метрике `accuracy` или нужно заранее планировать более строгую оценку.

In [ ]:
display_stage_note(
    "Шаг 3. Чтение нескольких форматов",
    "Сравним локальные тематические файлы, чтобы увидеть, как меняются функции чтения и структура результата.",
)

csv_fragment = pd.read_csv(FORMAT_DIR / "grid_stability_fragment.csv")
xlsx_fragment = pd.read_excel(FORMAT_DIR / "combined_cycle_power_plant_fragment.xlsx")
txt_fragment = pd.read_csv(FORMAT_DIR / "household_power_fragment.txt", sep=";")
json_catalog = pd.read_json(FORMAT_DIR / "dataset_catalog.json")

format_overview = pd.DataFrame(
    [
        {"Файл": "grid_stability_fragment.csv", "Формат": "CSV", "Строк": len(csv_fragment), "Столбцов": csv_fragment.shape[1], "Чтение": "pd.read_csv(...)"},
        {"Файл": "combined_cycle_power_plant_fragment.xlsx", "Формат": "XLSX", "Строк": len(xlsx_fragment), "Столбцов": xlsx_fragment.shape[1], "Чтение": "pd.read_excel(...)"},
        {"Файл": "household_power_fragment.txt", "Формат": "TXT", "Строк": len(txt_fragment), "Столбцов": txt_fragment.shape[1], "Чтение": "pd.read_csv(..., sep=';')"},
        {"Файл": "dataset_catalog.json", "Формат": "JSON", "Строк": len(json_catalog), "Столбцов": json_catalog.shape[1], "Чтение": "pd.read_json(...)"},
    ]
)
display_frame(format_overview, decimals=0)

for title, frame in [
    ("CSV: фрагмент набора устойчивости режима", csv_fragment.head(3)),
    ("XLSX: фрагмент набора по мощности станции", xlsx_fragment.head(3)),
    ("TXT: фрагмент временного ряда нагрузки", txt_fragment.head(3)),
    ("JSON: локальный каталог учебных файлов", json_catalog),
]:
    display(HTML(f"<h4 style='margin-top: 18px; color: #1f3556;'>{title}</h4>"))
    display_frame(frame)

## Что показывает сопоставление форматов

Различие форматов влияет на функцию чтения и иногда на параметры загрузки, но принцип анализа остается тем же. После открытия файла все равно нужно установить структуру таблицы, типы полей, границы периода наблюдений и роль каждого признака. Особенно важно различать данные наблюдений и метаданные: `JSON` в этом примере хранит не сами измерения, а описание учебных файлов.

In [ ]:
quality_issues = []
if duplicate_count:
    quality_issues.append("обнаружены дублирующиеся строки")
if profile["Доля пропусков"].max() > 0:
    quality_issues.append("обнаружены признаки с пропусками")
if not quality_issues:
    quality_issues.append("критических структурных нарушений в основном наборе не обнаружено")

display_callout(
    "Промежуточный вывод по качеству данных",
    "; ".join(quality_issues) + ". Основной набор можно использовать как отправную точку для следующих практикумов.",
    tone="success" if quality_issues == ["критических структурных нарушений в основном наборе не обнаружено"] else "warning",
)

## Итоговый вывод

Первичный контроль качества нужно воспринимать как обязательную аналитическую процедуру. На этом этапе выясняется, какие данные действительно готовы к исследованию, где возможны ошибки интерпретации и почему чтение разных форматов должно входить в базовый набор навыков специалиста по анализу энергетических данных.

## Вопросы для самостоятельной работы

1. Какие дополнительные проверки вы бы добавили, если бы основной набор содержал временные метки?
2. Почему файл `JSON` в практикуме лучше трактовать как метаданные, а не как таблицу измерений?
3. Чем рискованно без проверки загружать `TXT` как обычный `CSV`?

## Источники

1. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)
2. [Глава 2 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/02-data-sources-and-quality.md)
3. [Открытые датасеты UCI](https://archive.ics.uci.edu/)